# Complementary Visualisation Notebook — MD on Apocrita

**Purpose:** This notebook replaces all **VMD** and **xmgrace** visualisation steps from the `MD_On_Apocrita.md` tutorial with Python-based equivalents that run inside Jupyter on Apocrita via [OnDemand](https://docs.hpc.qmul.ac.uk/ondemand/jupyter/).

**How to use:**
1. Launch a Jupyter session on Apocrita through OnDemand. Ensure your tick the box to use a personal conda env. Then set the name of the personal conda env to be `GROMACS-tutorial`

2. Navigate to the Molecular_Dynamics_GROMACS directory. Inside you will find ./Tutorials/MD_On_Apocrita_Complementary_Notebook.ipynb. Open this! 

3. As you work through each section of the main tutorial in your terminal, come back here to run the corresponding visualisation cell.

**The notebook is organised to mirror the main tutorial sections:**
- Section 2 — Visualise the starting PDB structure
- Section 7 — Energy minimisation (potential energy plot)
- Section 8 — Equilibration (temperature, pressure, density plots)
- Section 10 — Analysis (trajectory viewer, RMSD, Rg, minimum distance, H-bonds)


---
## 0. Setup — Install Dependencies & Configure Paths

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# --------------------------------------------------------------------------
# CONFIGURE THIS: Set to the absolute path of your GROMACS data/ directory
# For example: /data/home/<username>/Learning_GROMACS/md-intro-tutorial-main/data
# --------------------------------------------------------------------------
DATA_DIR = os.path.expanduser("~/Learning_GROMACS/md-intro-tutorial-main/data")

os.chdir(DATA_DIR)
print(f"Working directory set to: {os.getcwd()}")

In [ ]:
def parse_xvg(filepath):
    """
    Parse a GROMACS .xvg file, skipping comment (#) and metadata (@) lines.
    Returns a dict with 'time' (first column) and 'columns' (remaining columns),
    plus 'labels' extracted from @ axis and legend metadata.
    """
    data_lines = []
    xlabel, ylabel = "Time (ps)", "Value"
    legend_labels = []

    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith("#"):
                continue
            elif line.startswith("@"):
                # Extract axis labels and legend entries
                if '"' in line:
                    label = line.split('"')[1]
                    if "xaxis" in line and "label" in line:
                        xlabel = label
                    elif "yaxis" in line and "label" in line:
                        ylabel = label
                    elif line.startswith("@ s") and "legend" in line:
                        legend_labels.append(label)
                continue
            else:
                vals = line.split()
                if vals:
                    data_lines.append([float(v) for v in vals])

    data = np.array(data_lines)
    return {
        "time": data[:, 0],
        "columns": data[:, 1:],
        "xlabel": xlabel,
        "ylabel": ylabel,
        "legend": legend_labels,
    }


def plot_xvg(filepath, title=None, figsize=(10, 4)):
    """
    Plot all data columns from an .xvg file with appropriate labels.
    """
    d = parse_xvg(filepath)
    fig, ax = plt.subplots(figsize=figsize)

    n_cols = d["columns"].shape[1]
    for i in range(n_cols):
        label = d["legend"][i] if i < len(d["legend"]) else f"Column {i+1}"
        ax.plot(d["time"], d["columns"][:, i], linewidth=0.8, label=label)

    ax.set_xlabel(d["xlabel"], fontsize=12)
    ax.set_ylabel(d["ylabel"], fontsize=12)
    if title:
        ax.set_title(title, fontsize=14)
    if n_cols > 1:
        ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return d


print("Helper functions loaded.")

---
## 2. Visualise the Starting PDB Structure

**Tutorial reference:** Section 2.1–2.2

After cleaning the PDB file (`1fjs_protein.pdb`), use the cell below to inspect the protein structure interactively in 3D — this replaces VMD.

In [ ]:
# Click and drag to rotate, zoom with your mousewheel 
# For more information about this viewer, have a look at https://github.com/nglviewer/nglview

import nglview as nv

view = nv.show_structure_file("input/1fjs.pdb")
view

In [ ]:
import nglview as nv

# Visualise the cleaned protein structure
view = nv.show_file("1fjs_protein.pdb")
view.clear_representations()
view.add_cartoon(selection="protein", color="chainid")
view.add_ball_and_stick(selection="hetero and not water", opacity=0.8)
view.center()
view

> **Tip:** Click and drag to rotate, scroll to zoom, right-click drag to translate. You can also try `view.add_surface(opacity=0.3)` to overlay a transparent surface.

---
## 5. Visualise the Solvated System

**Tutorial reference:** Section 5.2

After solvation, you can visualise the protein in its water box. Run this after completing Section 5 in the terminal.

In [ ]:
# Visualise the solvated system (protein + water box)
view_solv = nv.show_file("1fjs_solv.gro")
view_solv.clear_representations()
view_solv.add_cartoon(selection="protein", color="chainid")
view_solv.add_point(selection="water", color="blue", opacity=0.15)
view_solv.center()
view_solv

In [ ]:
# Visulaise the solvated system (protein + water box) after accounting for the rhombic dodecahedron shape.  (viz file)
 
view_solv = nv.show_file("1fjs_solv_viz.gro")
view_solv.clear_representations()
view_solv.add_cartoon(selection="protein", color="chainid")
view_solv.add_point(selection="water", color="blue", opacity=0.15)
view_solv.center()
view_solv

---
## 6. Visualise the Solvated + Ionised System

**Tutorial reference:** Section 6.3

After adding ions, visualise the final system with Na+ and Cl- ions. Run this after completing Section 6 in the terminal.

In [ ]:
# Visualise protein + ions (hide water for clarity)
view_ions = nv.show_file("1fjs_solv_ions.gro")
view_ions.clear_representations()
view_ions.add_cartoon(selection="protein", color="chainid")
view_ions.add_spacefill(selection=".NA", color="purple", radius=0.5)   # Na+ ions
view_ions.add_spacefill(selection=".CL", color="green", radius=0.5)    # Cl- ions
view_ions.center()
view_ions

In [ ]:
# Visualise protein + ions - Using adjusted output for visualisation
view_ions = nv.show_file("1fjs_solv_ions_viz.gro")
view_ions.clear_representations()
view_ions.add_cartoon(selection="protein", color="chainid")
view_ions.add_spacefill(selection=".NA", color="purple", radius=0.5)   # Na+ ions
view_ions.add_spacefill(selection=".CL", color="green", radius=0.5)    # Cl- ions
view_ions.center()
view_ions

---
## 7. Energy Minimisation — Potential Energy

**Tutorial reference:** Section 7.4 — *replaces `xmgrace potential.xvg`*

Run this cell after generating `potential.xvg` with:
```bash
printf "Potential\n0\n" | gmx energy -f em.edr -o potential.xvg
```

The plot should show **steady convergence** of the potential energy to a negative value (order of 10^5 kJ/mol).

In [ ]:
d = plot_xvg("potential.xvg", title="Energy Minimisation \u2014 Potential Energy")

# Print summary statistics
epot = d["columns"][:, 0]
print(f"\nInitial Epot: {epot[0]:,.0f} kJ/mol")
print(f"Final   Epot: {epot[-1]:,.0f} kJ/mol")
print(f"Change:       {epot[-1] - epot[0]:,.0f} kJ/mol")

---
## 8.1 NVT Equilibration — Temperature

**Tutorial reference:** Section 8.1 — *replaces `xmgrace temperature.xvg`*

Run this cell after generating `temperature.xvg` with:
```bash
echo "Temperature" | gmx energy -f nvt.edr -o temperature.xvg -b 20
```

The temperature should quickly reach **300 K** and remain stable.

In [ ]:
d = plot_xvg("temperature.xvg", title="NVT Equilibration \u2014 Temperature")

temp = d["columns"][:, 0]
print(f"\nMean temperature: {np.mean(temp):.1f} K")
print(f"Std deviation:    {np.std(temp):.1f} K")

# Add target temperature reference line
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(d["time"], temp, linewidth=0.8, label="Temperature")
ax.axhline(y=300, color="red", linestyle="--", alpha=0.6, label="Target (300 K)")
ax.set_xlabel(d["xlabel"], fontsize=12)
ax.set_ylabel("Temperature (K)", fontsize=12)
ax.set_title("NVT Equilibration \u2014 Temperature", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8.2 NPT Equilibration — Pressure

**Tutorial reference:** Section 8.2 — *replaces `xmgrace pressure.xvg`*

Run this cell after generating `pressure.xvg` with:
```bash
echo "Pressure" | gmx energy -f npt.edr -o pressure.xvg
```

Pressure will **fluctuate widely** (RMSE ~100 bar is normal). What matters is the *average* being close to **1 bar**.

In [ ]:
d = plot_xvg("pressure.xvg", title="NPT Equilibration \u2014 Pressure")

pressure = d["columns"][:, 0]
print(f"\nMean pressure:  {np.mean(pressure):.1f} bar")
print(f"Std deviation:  {np.std(pressure):.1f} bar")
print(f"RMSE from 1bar: {np.sqrt(np.mean((pressure - 1)**2)):.1f} bar")

# Running average to show convergence
window = max(1, len(pressure) // 20)
running_avg = np.convolve(pressure, np.ones(window)/window, mode="valid")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(d["time"], pressure, linewidth=0.3, alpha=0.5, label="Instantaneous")
ax.plot(d["time"][window-1:], running_avg, linewidth=1.5, color="red", label="Running average")
ax.axhline(y=1, color="black", linestyle="--", alpha=0.6, label="Target (1 bar)")
ax.set_xlabel(d["xlabel"], fontsize=12)
ax.set_ylabel("Pressure (bar)", fontsize=12)
ax.set_title("NPT Equilibration \u2014 Pressure (with running average)", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8.2 NPT Equilibration — Density

**Tutorial reference:** Section 8.2 — *replaces `xmgrace density.xvg`*

Run this cell after generating `density.xvg` with:
```bash
echo "Density" | gmx energy -f npt.edr -o density.xvg
```

The density should be close to **~1000 kg/m^3** (the density of liquid water).

In [ ]:
d = plot_xvg("density.xvg", title="NPT Equilibration \u2014 Density")

density = d["columns"][:, 0]
print(f"\nMean density:  {np.mean(density):.1f} kg/m\u00b3")
print(f"Std deviation: {np.std(density):.1f} kg/m\u00b3")

---
## 10.2 Visualise the MD Trajectory

**Tutorial reference:** Section 10.2 — *replaces VMD*

Run this cell after post-processing the trajectory (Section 10.1):
```bash
printf "1" | gmx trjconv -s md.tpr -f md.xtc -o md_whole.xtc -pbc whole
printf "1" | gmx trjconv -s md.tpr -f md_whole.xtc -o md_nojump.xtc -pbc nojump
printf "1\n1" | gmx trjconv -s md.tpr -f md_nojump.xtc -o md_center.xtc -center -pbc mol
```

This uses **NGLView** + **MDAnalysis** to display an interactive 3D trajectory viewer — you can scrub through frames using the playback controls.

In [ ]:
import MDAnalysis as mda
import nglview as nv

# Load the topology and centred trajectory
u = mda.Universe("1fjs_newbox.gro", "md_center.xtc")

# Create interactive trajectory viewer
view_traj = nv.show_mdanalysis(u)
view_traj.clear_representations()
view_traj.add_cartoon(selection="protein", color="chainid")
view_traj.center()

# Playback controls: use the slider beneath the viewer to scrub through frames
print(f"Trajectory: {u.trajectory.n_frames} frames, "
      f"{u.trajectory.dt} ps/frame, "
      f"total {u.trajectory.totaltime/1000:.1f} ns")
view_traj

> **Tip:** Use the play/pause button and slider below the viewer. You can also add representations interactively:
> ```python
> view_traj.add_surface(selection="protein", opacity=0.3, color="electrostatic")
> view_traj.add_licorice(selection="protein and sidechain", opacity=0.5)
> ```

---
## 10.3 Minimum Image Distance

**Tutorial reference:** Section 10.3 — *replaces `xmgrace mindist.xvg`*

Run this cell after generating `mindist.xvg` with:
```bash
printf "1\n" | gmx mindist -s md.tpr -f md_center.xtc -pi -od mindist.xvg
```

The minimum distance should **always exceed the non-bonded cutoff (1.2 nm)** — otherwise the protein is interacting with its own periodic image.

In [ ]:
d = plot_xvg("mindist.xvg", title="Minimum Periodic Image Distance")

mindist = d["columns"][:, 0]
print(f"\nMinimum distance observed: {np.min(mindist):.3f} nm")
print(f"Mean distance:             {np.mean(mindist):.3f} nm")

if np.min(mindist) < 1.2:
    print("\n\u26a0\ufe0f  WARNING: Distance dropped below 1.2 nm cutoff \u2014 periodic images may interact!")
    print("   Consider using a larger box (increase -d value in gmx editconf).")
else:
    print("\n\u2713 OK: Minimum distance always exceeds 1.2 nm cutoff.")

# Re-plot with cutoff line
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(d["time"], mindist, linewidth=0.8)
ax.axhline(y=1.2, color="red", linestyle="--", alpha=0.7, label="Non-bonded cutoff (1.2 nm)")
ax.set_xlabel(d["xlabel"], fontsize=12)
ax.set_ylabel("Distance (nm)", fontsize=12)
ax.set_title("Minimum Periodic Image Distance", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10.4 RMSD — Structural Stability

**Tutorial reference:** Section 10.4 — *replaces `xmgrace rmsd_xray.xvg`*

Run this cell after generating `rmsd_xray.xvg` with:
```bash
printf "4\n1\n" | gmx rms -s em.tpr -f md_center.xtc -o rmsd_xray.xvg -tu ns
```

The RMSD should level off around **~0.15 nm (1.5 \u00c5)**, indicating the protein has reached a stable conformation.

In [ ]:
d = plot_xvg("rmsd_xray.xvg", title="Backbone RMSD vs. Crystal Structure")

rmsd = d["columns"][:, 0]
print(f"\nFinal RMSD:         {rmsd[-1]:.3f} nm ({rmsd[-1]*10:.1f} \u00c5)")
print(f"Mean RMSD:          {np.mean(rmsd):.3f} nm ({np.mean(rmsd)*10:.1f} \u00c5)")
print(f"Mean (last 25%):    {np.mean(rmsd[len(rmsd)*3//4:]):.3f} nm "
      f"({np.mean(rmsd[len(rmsd)*3//4:])*10:.1f} \u00c5)")

---
## 10.5 Radius of Gyration — Compactness

**Tutorial reference:** Section 10.5 — *replaces `xmgrace gyrate.xvg`*

Run this cell after generating `gyrate.xvg` with:
```bash
echo "1" | gmx gyrate -f md_center.xtc -s md.tpr -o gyrate.xvg
```

A **stable Rg value** indicates the protein remains in its compact, folded form throughout the simulation.

In [ ]:
d = plot_xvg("gyrate.xvg", title="Radius of Gyration")

rg = d["columns"][:, 0]
print(f"\nMean Rg:          {np.mean(rg):.3f} nm")
print(f"Std deviation:    {np.std(rg):.3f} nm")
print(f"Mean Rg (last 25%): {np.mean(rg[len(rg)*3//4:]):.3f} nm")

---
## 10.6 Hydrogen Bonds Between Chains

**Tutorial reference:** Section 10.6 — *replaces `xmgrace hbnum.xvg`*

Run this cell after generating `hbnum.xvg` with:
```bash
printf "splitch 1\nq\n" | gmx make_ndx -f nvt.tpr -o chains.ndx
printf "17\n18\n" | gmx hbond -f md.xtc -s md.tpr -n chains.ndx -num hbnum.xvg
```

> **Note:** The group indices (17, 18) may differ \u2014 check the output of `gmx make_ndx` to confirm.

In [ ]:
d = plot_xvg("hbnum.xvg", title="Inter-Chain Hydrogen Bonds")

hbonds = d["columns"][:, 0]
print(f"\nMean H-bonds:  {np.mean(hbonds):.1f}")
print(f"Std deviation: {np.std(hbonds):.1f}")
print(f"Min / Max:     {np.min(hbonds):.0f} / {np.max(hbonds):.0f}")

---
## Summary Dashboard

Run this cell at the end to produce a combined overview of all analysis plots in a single figure.

In [ ]:
# Collect all available .xvg files for the summary dashboard
panels = [
    ("potential.xvg",    "Potential Energy"),
    ("temperature.xvg", "Temperature (NVT)"),
    ("pressure.xvg",    "Pressure (NPT)"),
    ("density.xvg",     "Density (NPT)"),
    ("rmsd_xray.xvg",   "RMSD vs. Crystal"),
    ("gyrate.xvg",      "Radius of Gyration"),
]

# Filter to only those that exist
available = [(f, t) for f, t in panels if os.path.isfile(f)]

if not available:
    print("No .xvg files found yet \u2014 run the GROMACS analysis commands first.")
else:
    n = len(available)
    cols = 2
    rows = (n + 1) // 2
    fig, axes = plt.subplots(rows, cols, figsize=(14, 3.5 * rows))
    axes = axes.flatten()

    for i, (filepath, title) in enumerate(available):
        d = parse_xvg(filepath)
        ax = axes[i]
        for j in range(d["columns"].shape[1]):
            label = d["legend"][j] if j < len(d["legend"]) else None
            ax.plot(d["time"], d["columns"][:, j], linewidth=0.6, label=label)
        ax.set_xlabel(d["xlabel"], fontsize=9)
        ax.set_ylabel(d["ylabel"], fontsize=9)
        ax.set_title(title, fontsize=11, fontweight="bold")
        if d["legend"]:
            ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=8)

    # Hide any empty subplot slots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("MD Simulation \u2014 Analysis Dashboard", fontsize=15, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("analysis_dashboard.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\nDashboard saved to analysis_dashboard.png")

---
## Notes

- **NGLView** replaces VMD for interactive 3D molecular visualisation.
- **Matplotlib** replaces xmgrace for all 2D plots from `.xvg` files.
- **MDAnalysis** is used to load GROMACS trajectories into NGLView.
- All plots are generated from the same `.xvg` files that the main tutorial produces \u2014 this notebook simply provides a Python-based viewer.
- To launch Jupyter on Apocrita, see the [OnDemand documentation](https://docs.hpc.qmul.ac.uk/ondemand/jupyter/).